In [9]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Set date range for 2 years of daily observations
date_range = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')

# Generate random data
np.random.seed(0)
data = np.random.randn(len(date_range)) * 10 + 100  # Base mean around 100 with some random noise

# Create DataFrame
time_series_df = pd.DataFrame({'date': date_range, 'value': data})

# Extract month number and calculate t and T
time_series_df['month'] = time_series_df['date'].dt.month
T = 12  # Total number of months (2 years)

# Add sine and cosine components
time_series_df['sine_component'] = np.sin(2 * np.pi * time_series_df['month'] / T)
time_series_df['cosine_component'] = np.cos(2 * np.pi * time_series_df['month'] / T)

# Add trend and trend squared
time_series_df['trend'] = np.arange(len(time_series_df))  # Linear trend
time_series_df['trend_squared'] = time_series_df['trend'] ** 2  # Quadratic trend

# Prepare the regression model
X = time_series_df[['sine_component', 'cosine_component', 'trend', 'trend_squared']]
X = sm.add_constant(X)  # Add constant for the intercept
y = time_series_df['value']

# Fit the full regression model
model = sm.OLS(y, X).fit()

# Perform the F-test for sine and cosine significance
hypothesis_test_sine_cosine = model.f_test("sine_component = 0, cosine_component = 0")
print("F-test for sine and cosine significance:")
print(hypothesis_test_sine_cosine)

# Perform the F-test for trend and trend_squared significance
hypothesis_test_trend = model.f_test("trend = 0, trend_squared = 0")
print("\nF-test for trend and trend_squared significance:")
print(hypothesis_test_trend)



F-test for sine and cosine significance:
<F test: F=0.6723768048462878, p=0.5108118267724018, df_denom=725, df_num=2>

F-test for trend and trend_squared significance:
<F test: F=1.688949783899133, p=0.1854393491960464, df_denom=725, df_num=2>


In [22]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Set parameters
num_items = 5  # Number of items
date_range = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
T = 12  # Total number of months (2 years)

# Create an empty list to store data for each item
data = []

# Generate synthetic data for each item
np.random.seed(0)
for item in range(1, num_items + 1):
    base_values = np.random.randn(len(date_range)) * 5 + 100  # Base values with random noise
    
    # Apply seasonality and trend selectively
    if item in [1, 2]:  # Items with seasonality
        seasonal_values = 10 * np.sin(2 * np.pi * date_range.month / T) + 8 * np.cos(2 * np.pi * date_range.month / T)
        values = base_values + seasonal_values
    elif item in [3, 4]:  # Items with trend
        trend_values = np.linspace(0, 50, len(date_range))  # Linear trend over time
        trend_squared_values = np.linspace(0, 25, len(date_range))**2  # Quadratic trend over time
        values = base_values + trend_values + trend_squared_values * 0.01
    else:  # Item with neither trend nor seasonality (control group)
        values = base_values
    
    # Create DataFrame for each item
    df = pd.DataFrame({
        'item': item,
        'date': date_range,
        'value': values
    })
    
    # Calculate month, sine, cosine, trend, and trend squared components for testing purposes
    df['month'] = df['date'].dt.month
    df['sine_component'] = np.sin(2 * np.pi * df['month'] / T)
    df['cosine_component'] = np.cos(2 * np.pi * df['month'] / T)
    df['trend'] = np.arange(len(df))  # Linear trend
    df['trend_squared'] = df['trend'] ** 2  # Quadratic trend
    
    # Append to the data list
    data.append(df)

# Combine all items into a single DataFrame
panel_data = pd.concat(data, ignore_index=True)

# Prepare a list to store F-test results
f_test_results = []

# Run regression and F-tests for each item
for item in panel_data['item'].unique():
    item_data = panel_data[panel_data['item'] == item]
    
    # Prepare the regression model
    X = item_data[['sine_component', 'cosine_component', 'trend', 'trend_squared']]
    X = sm.add_constant(X)  # Add constant for the intercept
    y = item_data['value']
    
    # Fit the full regression model
    model = sm.OLS(y, X).fit()
    
    # Perform the F-test for sine and cosine significance
    f_test_sine_cosine = model.f_test("sine_component = 0, cosine_component = 0")
    sine_cosine_stat = round(f_test_sine_cosine.fvalue, 3)
    sine_cosine_pvalue = round(f_test_sine_cosine.pvalue, 3)
    
    # Perform the F-test for trend and trend_squared significance
    f_test_trend = model.f_test("trend = 0, trend_squared = 0")
    trend_stat = round(f_test_trend.fvalue, 3)
    trend_pvalue = round(f_test_trend.pvalue, 3)
    
    # Store the results
    f_test_results.append({
        'item': item,
        
        'sine_cosine_p_value': sine_cosine_pvalue,
       
        'trend_p_value': trend_pvalue
    })

# Convert results to DataFrame
f_test_results_df = pd.DataFrame(f_test_results)

# Display the results with specified decimal format
print(f_test_results_df.to_string(index=False))


 item  sine_cosine_p_value  trend_p_value
    1                0.000          0.185
    2                0.000          0.100
    3                0.375          0.000
    4                0.873          0.000
    5                0.816          0.247


In [23]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller  # Import ADF test

# Set parameters
num_items = 5  # Number of items
date_range = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')
T = 12  # Total number of months (2 years)

# Create an empty list to store data for each item
data = []

# Generate synthetic data for each item
np.random.seed(0)
for item in range(1, num_items + 1):
    base_values = np.random.randn(len(date_range)) * 5 + 100  # Base values with random noise
    
    # Apply seasonality and trend selectively
    if item in [1, 2]:  # Items with seasonality
        seasonal_values = 10 * np.sin(2 * np.pi * date_range.month / T) + 8 * np.cos(2 * np.pi * date_range.month / T)
        values = base_values + seasonal_values
    elif item in [3, 4]:  # Items with trend
        trend_values = np.linspace(0, 50, len(date_range))  # Linear trend over time
        trend_squared_values = np.linspace(0, 25, len(date_range))**2  # Quadratic trend over time
        values = base_values + trend_values + trend_squared_values * 0.01
    else:  # Item with neither trend nor seasonality (control group)
        values = base_values
    
    # Create DataFrame for each item
    df = pd.DataFrame({
        'item': item,
        'date': date_range,
        'value': values
    })
    
    # Calculate month, sine, cosine, trend, and trend squared components for testing purposes
    df['month'] = df['date'].dt.month
    df['sine_component'] = np.sin(2 * np.pi * df['month'] / T)
    df['cosine_component'] = np.cos(2 * np.pi * df['month'] / T)
    df['trend'] = np.arange(len(df))  # Linear trend
    df['trend_squared'] = df['trend'] ** 2  # Quadratic trend
    
    # Append to the data list
    data.append(df)

# Combine all items into a single DataFrame
panel_data = pd.concat(data, ignore_index=True)

# Prepare a list to store F-test results
f_test_results = []

# Run regression, F-tests, and ADF test for each item
for item in panel_data['item'].unique():
    item_data = panel_data[panel_data['item'] == item]
    
    # Prepare the regression model
    X = item_data[['sine_component', 'cosine_component', 'trend', 'trend_squared']]
    X = sm.add_constant(X)  # Add constant for the intercept
    y = item_data['value']
    
    # Fit the full regression model
    model = sm.OLS(y, X).fit()
    
    # Perform the F-test for sine and cosine significance
    f_test_sine_cosine = model.f_test("sine_component = 0, cosine_component = 0")
    sine_cosine_stat = round(f_test_sine_cosine.fvalue, 3)
    sine_cosine_pvalue = round(f_test_sine_cosine.pvalue, 3)
    
    # Perform the F-test for trend and trend_squared significance
    f_test_trend = model.f_test("trend = 0, trend_squared = 0")
    trend_stat = round(f_test_trend.fvalue, 3)
    trend_pvalue = round(f_test_trend.pvalue, 3)
    
    # Run ADF test
    adf_result = adfuller(item_data['value'])
    adf_pvalue = round(adf_result[1], 3)  # ADF test p-value
    
    # Store the results
    f_test_results.append({
        'item': item,
        'sine_cosine_p_value': sine_cosine_pvalue,
        'trend_p_value': trend_pvalue,
        'adf_p_value': adf_pvalue  # Add ADF p-value to results
    })

# Convert results to DataFrame
f_test_results_df = pd.DataFrame(f_test_results)

# Display the results with specified decimal format
print(f_test_results_df.to_string(index=False))


 item  sine_cosine_p_value  trend_p_value  adf_p_value
    1                0.000          0.185        0.324
    2                0.000          0.100        0.441
    3                0.375          0.000        0.912
    4                0.873          0.000        0.934
    5                0.816          0.247        0.000
